[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-2_%EC%9D%B4%EC%83%81%ED%83%90%EC%A7%80_%EC%A0%95%EC%83%81%EC%9D%84%EB%B0%B0%EC%9A%B0%EA%B3%A0_%EB%B2%97%EC%96%B4%EB%82%A8%EC%B0%BE%EA%B8%B0.ipynb)

# 2-2 · 이상탐지 — 정상 데이터만으로 경보 만들기

**⏱ 100분** · 모듈 2: 센서 → 특징 → **① 이상탐지** → 고장진단 → 판정 기준

## 🎯 이번 시간의 질문

> "정상이 어떤 모습인지"만 알고 있을 때, **처음 보는 고장**에 경보를 울릴 수 있을까요?

1-2에서는 "보류 섹터 > 0"이라는 기준을 바로 쓸 수 있었습니다. 정상 디스크는 0이라는 것을 알았기 때문입니다.
그런데 유압 장치의 오일 온도는 **몇 도가 정상**일까요? 알 수 없습니다. 게다가 1-1에서 본 것처럼 현장에는 고장 기록이 드뭅니다.
그래서 **정상일 때의 데이터에서 기준을 배우는** 방법을 씁니다. 이것이 **이상탐지(Anomaly Detection)**입니다.

```
① 정상일 때의 데이터로 "평소 모습"을 배운다 → ② 새 데이터가 평소에서 얼마나 먼지 점수를 매긴다 → ③ 점수가 기준을 넘으면 경보
```

2-1의 구분력 표를 기억하며 두 경우를 비교합니다.

| | 대상 | 2-1 구분력 | 방법 |
|---|---|---|---|
| 쉬운 경우 | 냉각기 (100% = 정상) | 13 | 방법 1 · z-점수 (센서 1개) |
| 어려운 경우 | 펌프 누설 (0 = 정상) | 1.1 | 먼저 z-점수를 그대로 → 안 되면 방법 2 · Isolation Forest (센서 14개를 함께) |

**이번 시간에 익힐 패턴:** P2 조건으로 나누기 · 평균·표준편차로 점수 만들기 · P5 `fit` (정답 없이 학습) · 교차표로 평가 · 겹친 히스토그램

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
labels = pd.read_csv('data/hydraulic_labels.csv')
features = pd.read_csv('data/hydraulic_features.csv')
data = labels.merge(features, on='cycle_id')      # 2-1 5절과 같은 합친 표: 한 행 = 사이클 1번
print('읽기 완료:', data.shape)

## 1. 학습용과 평가용, 그리고 "정상" 데이터

- `train`(학습용): 기준을 배우는 데 씁니다. 여기서 **냉각기가 정상(100%)인 사이클만** 골라 `normal`로 둡니다.
- `test`(평가용): 기준을 정한 **뒤에** 성능을 확인하는 데만 씁니다. 배운 데이터로 평가하면 성능이 부풀려지기 때문입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
train = data[data['split'] == 'train']
test = data[data['split'] == 'test']
normal = train[train['cooler'] == 100]
print('정상 학습 사이클:', len(normal))
print('평가 사이클:', len(test))
test['cooler'].value_counts()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 정상 학습 사이클 629개로 "평소 모습"을 배우고, 처음 보는 평가 사이클 336개(냉각기 상태별 112개)로 확인합니다.

### "정상" 사이클 629개는 정말 정상일까?

`normal`은 **냉각기가 100%인** 사이클입니다. 2-1에서 네 부품 상태의 조합 144가지를 모두 시험했다고 했으니, 나머지 세 부품은 어떤 상태일까요? 2-1의 `value_counts` 반복을 `normal`에 그대로 씁니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for part in ['valve', 'pump_leakage', 'accumulator', 'stable_flag']:
    print(normal[part].value_counts().sort_index())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — `normal` 629개 중 밸브가 정상(100)인 것은 359개, 누설이 없는(0) 것은 383개, 축압기가 정상(130)인 것은 113개뿐입니다. **네 부품이 모두 정상인 사이클은 21개**입니다. 상태를 바꾼 직후의 과도 구간(`stable_flag = 1`)도 250개 들어 있습니다.
즉 이 "정상"은 "장치 전체가 정상"이 아니라 **"냉각기만 정상"**입니다. 밸브 지연·누설·축압기 저압은 냉각기를 감시하는 입장에서는 **평소의 변동**으로 보고 함께 배웁니다. 현장에서도 감시 대상 부품 말고 다른 부품의 상태는 늘 조금씩 다르므로, 이렇게 배우는 것이 현실에 가깝습니다. 이 점이 3절에서 펌프 누설이 잘 안 잡히는 이유가 됩니다.

---
## 2. 방법 1 · z-점수로 냉각기 감시

2-1에서 오일 온도 `TS1_mean`이 냉각기 상태를 잘 보여 줬습니다. 정상 온도의 평균과 흩어짐을 배웁니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
mu = normal['TS1_mean'].mean()
sigma = normal['TS1_mean'].std()
print('정상 평균:', round(mu, 2), '°C')
print('정상 표준편차:', round(sigma, 2), '°C')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 정상 평균 35.94°C, 표준편차 0.72°C.

이 두 숫자로 **"평균 ± 3 × 표준편차"** 선을 그어 온도를 **시간 순서대로** 보면, 품질관리에서 배운 **관리도**가 됩니다. (▶ 2,205 사이클 전체를 시간 순서로 그립니다. 2-1의 실험 일정표와 비교해 보세요)

In [ ]:
# ▶ 그대로 실행하세요
plt.figure(figsize=(11, 4))
for state, color in [(100, 'C0'), (20, 'C1'), (3, 'C3')]:
    part = data[data['cooler'] == state]
    plt.scatter(part['cycle_id'], part['TS1_mean'], s=3, color=color, label=f'냉각기 {state}%')
plt.axhline(mu + 3 * sigma, color='black', linestyle='--', label='정상 평균 ± 3σ')
plt.axhline(mu - 3 * sigma, color='black', linestyle='--')
plt.xlabel('사이클 번호 (시간 순서)')
plt.ylabel('오일 온도 TS1 평균 (°C)')
plt.legend(markerscale=4)
plt.show()

💬 **결과 해설** — 이 장치는 냉각기 3% → 20% → 100% 순서로 시험했습니다. 냉각기가 정상(파랑)인 구간은 ±3σ 띠 안에 있고, 냉각기가 나빠진 구간은 띠보다 **6~9°C(20%)**, **12~20°C(3%)** 높습니다.
눈여겨볼 곳이 두 군데 있습니다. 시험 시작 직후(1~30번)는 기름이 아직 데워지는 중이라 냉각기가 3%인데도 띠 근처에 있고, 1,465번 직후는 냉각기를 고친 뒤 기름이 식는 중이라 정상인데도 띠 위에 있습니다. 상태가 바뀐 직후의 **과도 구간**에서는 경보가 늦거나 잘못 울릴 수 있습니다.

### z-점수 = "정상 평균에서 표준편차 몇 개만큼 떨어졌나"

$$z = \frac{\text{새 값} - \text{정상 평균}}{\text{정상 표준편차}}$$

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
z = (test['TS1_mean'] - mu) / sigma
for state in [100, 20, 3]:
    plt.hist(z[test['cooler'] == state], bins=20, alpha=0.6, label=f'냉각기 {state}%')
plt.axvline(3, color='red', linestyle='--', label='기준 z = 3')
plt.xlabel('z-점수')
plt.ylabel('사이클 수')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 냉각기 정상(100%)은 z가 0 근처, 효율 저하(20%)는 약 12, 거의 고장(3%)은 약 28입니다. 기준 z = 3이 정상과 이상 사이를 깔끔하게 가릅니다.

### 경보를 평가하는 네 칸 — 1-2와 같은 방법

|z| > 3이면 경보. **실제로 이상인가 × 경보가 울렸나** 교차표를 만듭니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
alarm = z.abs() > 3
is_fault = test['cooler'] != 100
pd.crosstab(is_fault, alarm, rownames=['실제 이상'], colnames=['경보'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


1-2의 네 칸과 똑같이 읽습니다: 이상인데 경보(**탐지**), 이상인데 경보 없음(**놓침**), 정상인데 경보(**헛경보**), 정상이고 경보 없음(정상).

두 지표를 계산합니다.
- **탐지율** = 탐지 ÷ 실제 이상 → 1-2와 같음
- **오경보율** = 헛경보 ÷ 실제 정상 → "정상 사이클 중 몇 %에 괜히 경보가 울렸나"

1-2에서 쓴 적중률(경보 중 실제 고장 비율) 대신 오경보율을 쓰는 이유: 이 시험 데이터는 평가용의 2/3가 이상 사이클이라 적중률이 현장보다 훨씬 높게 나옵니다. 현장처럼 고장이 드물 때의 부담은 "정상인데 울린 비율"로 보는 것이 정확합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
hits = (alarm & is_fault).sum()
false_alarms = (alarm & ~is_fault).sum()
print('탐지율:', round(hits / is_fault.sum() * 100, 1), '%')
print('오경보율:', round(false_alarms / (~is_fault).sum() * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 탐지율 100.0%, 오경보율 0.0%. (`~`는 "아니다": `~is_fault` = 실제 정상)

💬 **결과 해설** — 정상 데이터 629개만으로 만든 기준이 처음 보는 냉각기 이상 224개를 **모두** 잡고, 정상 112개에는 한 번도 울리지 않았습니다.
모델이 뛰어나서가 아니라 **특징이 좋아서**입니다(구분력 13). 그렇다면 구분력이 1밖에 안 되는 펌프 누설은 어떨까요?

### ▶ 요약 함수

3절부터는 같은 평가를 반복하므로 한 줄로 묶었습니다(1-2의 `summarize`와 같은 역할).

In [ ]:
# ▶ 그대로 실행하세요
def summarize(alarm, is_fault):
    """경보 결과 한 줄 요약: 경보 수, 탐지, 탐지율, 헛경보, 오경보율."""
    alarm, is_fault = np.asarray(alarm), np.asarray(is_fault)
    hits = (alarm & is_fault).sum()
    false_alarms = (alarm & ~is_fault).sum()
    return pd.Series({'경보 수': alarm.sum(), '탐지': hits,
                      '탐지율(%)': round(hits / is_fault.sum() * 100, 1),
                      '헛경보': false_alarms,
                      '오경보율(%)': round(false_alarms / (~is_fault).sum() * 100, 1)})

summarize(alarm, is_fault)

---
## 3. 펌프 누설 감시 — z-점수를 그대로 써 보고, 안 되면 센서 14개를 함께

### 먼저 같은 방법으로 — 전동기 전력의 z-점수

2-1 과제 2에서 펌프 누설의 1위 특징은 **전동기 전력 `EPS1_mean`**(구분력 1.08)이었습니다. 누설이 있으면 같은 일을 하려고 전력을 더 쓰기 때문입니다. 2절의 z-점수를 **그대로** 반복합니다. 정상은 **누설이 없는(0) 학습 사이클**, 전력이 올라가면 이상이므로 z > 3이면 경보입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
normal_p = train[train['pump_leakage'] == 0]
is_leak = test['pump_leakage'] > 0
z_p = (test['EPS1_mean'] - normal_p['EPS1_mean'].mean()) / normal_p['EPS1_mean'].std()
print('정상 학습 사이클:', len(normal_p), '/ 평가: 누설', is_leak.sum(), '· 정상', (~is_leak).sum())
summarize(z_p > 3, is_leak)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 정상 학습 사이클 1,131, 평가 누설 246 · 정상 90, 경보 3, 탐지 3, **탐지율 1.2%**, 헛경보 0.

💬 누설 246사이클 중 **3개만** 잡았습니다. 냉각기(100%)와 왜 이렇게 다를까요? 1절에서 본 것처럼 `normal_p`(누설 0)에도 다른 부품 상태가 섞여 있습니다. **P3 · 묶어 요약**으로 누설 없는 사이클의 전력을 냉각기 상태별로 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
normal_p.groupby('cooler')['EPS1_mean'].mean()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 누설이 **없는** 사이클인데도 냉각기 상태에 따라 전력이 약 **2,409 · 2,451 · 2,541 W**로, 130 W 넘게 다릅니다(기름 온도가 다르면 기름의 끈적임이 달라져 전동기 부하가 달라짐). 반면 누설 때문에 오르는 전력은 없음 → 약함 → 심함에서 약 2,468 → 2,519 → 2,541 W, **50~75 W**입니다(2-1 과제 2의 상자그림).
**다른 부품이 만든 흩어짐이 누설 신호보다 커서**, 전력 하나로는 누설이 그 안에 묻힙니다. 이것이 구분력 1.1의 실제 뜻입니다.
센서 하나로 안 된다면, **센서 여러 개를 함께** 보면 다른 부품의 영향과 누설을 구분할 수 있을까요?

### 방법 2 · 센서 14개를 함께 — Isolation Forest

입력으로 쓸 특징을 정합니다. 특징 표의 70열은 **센서 14개 × 요약 5개**입니다. 여기서는 센서마다 **60초 평균(`_mean`) 하나씩, 14개**를 씁니다. 평균은 그 사이클 동안 센서의 전체 수준을 가장 직접 보여 주고, 5가지 요약을 모두 넣는 것보다 무엇을 보고 판단했는지 설명하기 쉽습니다(2-3 진단에서도 같은 14개를 씁니다).

In [ ]:
# ▶ 그대로 실행하세요
mean_cols = ['PS1_mean', 'PS2_mean', 'PS3_mean', 'PS4_mean', 'PS5_mean', 'PS6_mean', 'EPS1_mean',
             'FS1_mean', 'FS2_mean', 'TS1_mean', 'TS2_mean', 'TS3_mean', 'TS4_mean', 'VS1_mean']
print('입력 특징', len(mean_cols), '개:', mean_cols)

센서가 여러 개면 "평균에서 몇 σ"를 하나로 정할 수 없으므로 머신러닝 이상탐지를 씁니다.

**Isolation Forest의 아이디어** — 데이터 공간에 무작위로 칸막이를 쳐서 점 하나를 혼자 남게(고립) 만든다고 생각해 봅시다.
정상 무리 한가운데 있는 점은 칸막이를 아주 많이 쳐야 고립되지만, **무리에서 떨어진 점은 몇 번 만에 고립**됩니다. "쉽게 고립되는 점일수록 이상"이라고 보고, 무작위 나무 100그루의 평균으로 점수를 매깁니다.

정상 데이터는 z-점수 때와 같은 `normal_p`(누설이 없는 학습 사이클 1,131개)입니다.

**P5 · 모델 학습 — 정답(y) 없이 `fit`**. 정상 데이터(X)만 넣으므로 **비지도 학습**이라고 부릅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
from sklearn.ensemble import IsolationForest

iso = IsolationForest(random_state=0)
iso.fit(normal_p[mean_cols])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


### 이상 점수와 기준

- `score_samples()`는 **정상다울수록 큰 값**을 줍니다. 앞에 `-`를 붙여 **클수록 이상**한 점수로 바꿉니다.
- 기준은 이상 데이터 없이 **정상 데이터의 점수 분포**로 정합니다. z > 3도 "정상이 거의 넘지 않는 선"이었습니다. 이번 점수는 센서 14개를 합친 값이라 표준편차로 선을 긋기 어려우므로, 그 선을 직접 정합니다. `np.quantile(점수, 0.99)` = 정상 학습 사이클의 99%가 이 값보다 작다 → "정상도 100번 중 1번은 넘는 선"입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
normal_score = -iso.score_samples(normal_p[mean_cols])
test_score = -iso.score_samples(test[mean_cols])
threshold = np.quantile(normal_score, 0.99)
alarm_p = test_score > threshold
summarize(alarm_p, is_leak)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 경보 51, 탐지 50, **탐지율 20.3%**, 헛경보 1, 오경보율 1.1%.

💬 **결과 해설** — 센서 하나(1.2%)보다는 훨씬 낫지만, 센서 14개를 함께 봐도 누설 246개 중 **50개(20%)만** 잡았습니다. 왜 그런지 점수 분포를 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for state in [0, 1, 2]:
    plt.hist(test_score[test['pump_leakage'] == state], bins=30, alpha=0.6, label=f'누설 {state}')
plt.axvline(threshold, color='red', linestyle='--', label='기준 (정상 99%)')
plt.xlabel('이상 점수 (클수록 이상)')
plt.ylabel('사이클 수')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 누설 사이클(1, 2)의 점수가 정상(0)보다 **조금 오른쪽에** 있지만 대부분 같은 범위에 겹칩니다. 전력 하나에서 본 것과 같은 이유입니다. "정상"으로 배운 `normal_p`에 냉각기 3%·20%·100%, 밸브 지연, 축압기 저압이 모두 들어 있어서 14개 센서 값이 이미 넓게 퍼져 있고, 누설이 만드는 작은 변화(전력 50~75 W 등)는 그 퍼짐 안에 묻힙니다. 정상 데이터만으로는 "다른 부품 때문에 달라진 것"과 "누설 때문에 달라진 것"을 구분할 방법이 없습니다.
기준(빨간 선)을 **왼쪽으로 옮기면** 누설을 더 잡겠지만, 정상 사이클에도 경보가 늘어납니다. 얼마나 늘까요? → 과제 1.

### ▶ 이상탐지 함수

3절의 과정(정상 학습 → 점수 → 기준 → 요약)을 한 줄로 묶었습니다. `anomaly_check(부품, 정상 상태값, q=기준 분위수)`

In [ ]:
# ▶ 그대로 실행하세요
def anomaly_check(part, normal_state, q=0.99):
    """part가 normal_state인 학습 사이클로 Isolation Forest를 학습하고, 평가 사이클의 경보 결과를 요약한다."""
    normal = train[train[part] == normal_state]
    model = IsolationForest(random_state=0).fit(normal[mean_cols])
    th = np.quantile(-model.score_samples(normal[mean_cols]), q)
    alarm = -model.score_samples(test[mean_cols]) > th
    return summarize(alarm, test[part] != normal_state)

anomaly_check('pump_leakage', 0)

✅ 3절과 같은 결과(탐지 50, 탐지율 20.3%)가 나오면 됩니다.

---
## 4. 실습 과제 (40분)

### 📝 과제 1 · 기준을 낮추면 누설을 더 잡을 수 있나? (기준값 실험)
기준 분위수 q를 0.99에서 **0.95, 0.9, 0.8, 0.7**로 낮추며 표를 채우세요. **실행 전에** 탐지율과 오경보율이 각각 어떻게 변할지 예상해 보세요.

| q | 경보 수 | 탐지 | 탐지율(%) | 헛경보 | 오경보율(%) |
|---|---|---|---|---|---|
| 0.99 | 51 | 50 | 20.3 | 1 | 1.1 |
| 0.95 | | | | | |
| 0.9 | | | | | |
| 0.8 | | | | | |
| 0.7 | | | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: anomaly_check('pump_leakage', 0, q=0.95) 처럼 q만 바꿉니다.


✏️ **나의 답:** 누설의 80% 이상을 잡으려면 정상 사이클의 몇 %에 경보를 감수해야 하나요? 현장에서 받아들일 수 있을까요?

### 📝 과제 2 · 부품별로 이상탐지가 얼마나 되나?
네 부품 각각의 정상 상태로 이상탐지를 해 보고, 2-1의 구분력과 나란히 표를 채우세요(q = 0.99).

| 부품 | 정상 상태값 | 2-1 최고 구분력 | 탐지율(%) | 오경보율(%) |
|---|---|---|---|---|
| 냉각기 | 100 | 13.2 | | |
| 펌프 | 0 | 1.1 | 20.3 | 1.1 |
| 밸브 | 100 | 1.2 | | |
| 축압기 | 130 | 1.2 | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: anomaly_check('cooler', 100), anomaly_check('valve', 100), anomaly_check('accumulator', 130)


✏️ **나의 답:** 구분력과 탐지율 사이에 어떤 관계가 보이나요?

### 📝 과제 3 · 생각해 보기
1) 이상탐지 경보가 울렸을 때 정비원이 **알 수 있는 것**과 **알 수 없는 것**을 한 가지씩 적으세요.
2) 이상탐지는 고장 데이터 없이도 쓸 수 있다는 큰 장점이 있습니다. 현장에서 이상탐지를 먼저 도입한다면, **어떤 부품·고장부터** 적용하겠습니까? 과제 2의 표를 근거로 답하세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 이상탐지의 원리 | 정상 데이터로 기준 학습 → 점수 → 기준 넘으면 경보. **고장 데이터가 없어도** 된다 |
| "정상"의 뜻 | 냉각기 감시의 정상 629개 중 네 부품이 모두 정상인 것은 21개. 다른 부품의 변동까지 **평소 모습**으로 배운다 |
| z-점수 | (값 − 정상 평균) ÷ 정상 표준편차. 시간 순서로 그리면 관리도. 냉각기 100% · 펌프 누설(전력) 1.2% |
| Isolation Forest | 여러 센서를 함께 보고 "무리에서 떨어진 정도"로 점수. `fit(X)`만, 정답 없음 |
| 평가 | 1-2와 같은 네 칸 → 탐지율·오경보율. `summarize`, `anomaly_check`로 반복 |
| 교환 관계 | 기준을 낮추면 탐지율↑ 오경보율↑ (펌프: 20% / 1% ↔ 86% / 29%) |
| 한계 | 다른 부품이 만든 흩어짐(냉각기 상태별 전력 130 W)이 누설 신호(50~75 W)보다 크면 안 잡힌다. 경보가 **무엇이** 고장인지 알려 주지 않는다 |

다음 질문: 과거 기록에 "이 사이클은 누설 약함이었다"는 **정답**이 있다면, 센서 14개로 누설을 판정할 수 있을까요? → **2-3 · 고장진단**